In [1]:
from srsinst.sr860 import SR860
import pyvisa
import serial
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
rm=pyvisa.ResourceManager()
print("Available VISA resources: ")
print(rm.list_resources())

Available VISA resources: 
('USB0::0xB506::0x2000::006521::INSTR', 'USB0::0x05E6::0x6500::MYFP001475::INSTR', 'ASRL1::INSTR', 'ASRL3::INSTR')


In [3]:
try:
    lockin = SR860('visa', 'USB0::0xB506::0x2000::006521::INSTR')
    device_id = lockin.check_id()
    print(f"SR860 Connected: {device_id}")
    print(f"Status: {lockin.get_status()}")
except Exception as e:
    print(f"Error connecting to SR860: {e}")

SR860 Connected: ('SR860', '006521', '1.55')
Status: Frequency: 4.504841e-04 Hz
 Phase: 96.385755 deg
 Amplitude: 0.5000 V
 DC level: 0.0000 V
 Harmonic: 1

LIA status bit 3, UNLK is set, Event status bit 6, URQ is set, Event status bit 7, PON is set, Overload bit 3, external reference unlocked, is set,


In [4]:
lockin.reset()
lockin.ref.sine_out_amplitude=0.5 #setting as 500mV
print(f"Sine output amplitude:{lockin.ref.sine_out_amplitude} V")
lockin.ref.auto_phase()
print("Auto-phase completed")

Sine output amplitude:0.5 V
Auto-phase completed


In [5]:
import sys
import time
import os
import serial.tools.list_ports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
sys.path.append(r"C:\Users\dell_7020_6\Desktop\holmarc electromagnet")

In [6]:
from Class.DeviceInterface import DeviceInterface
from Class.SettingsManager import SettingsManager
from Class.DeviceController import deviceControl

print("Holmarc electromagnet modules imported successfully")

Holmarc electromagnet modules imported successfully


In [7]:
class ElectromagnetController:
    def __init__(self, com_port):
        self.com_port = com_port
        self.connected = False

    def connect(self):
        # Load vendor settings (serial speed, limits, etc.)
        SettingsManager.load_settings()

        # Configure serial port
        DeviceInterface.config_serialport(self.com_port)

        # Connect to electromagnet
        if DeviceInterface.connect():
            self.connected = True
            print(f"Electromagnet connected on {self.com_port}")
        else:
            raise RuntimeError("Failed to connect electromagnet")

    def set_current(self, current_a, wait_time=2.0):
    # Safety limit
        if abs(current_a) > 0.001:
            raise ValueError("Max allowed current is 1 mA")

    # Step 1: Explicitly enable electromagnet output
        DeviceInterface.electromagnet_on()
        time.sleep(0.5)

    # Step 2: Force Constant Current mode (reset first)
        DeviceInterface.set_field("Constant Current", 0.0)
        time.sleep(0.5)

    # Step 3: Apply the desired current
        DeviceInterface.set_field("Constant Current", current_a)
        time.sleep(wait_time)

        print(f"Magnet current command sent: {current_a*1e3:.3f} mA")


    def read_field(self):
        """
        Read magnetic field in mT
        """
        DeviceInterface.read_magnetic_field()
        return deviceControl.x_moved

    def off(self):
        DeviceInterface.electromagnet_off()
        print("Electromagnet turned OFF")



In [8]:
print("\nAvailable COM ports:")
ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(p)


Available COM ports:
COM3 - USB Serial Port (COM3)
COM1 - Communications Port (COM1)


In [9]:
MAGNET_COM_PORT = "COM3"

In [10]:
magnet = ElectromagnetController(MAGNET_COM_PORT)
magnet.connect()

Electromagnet connected on COM3


In [11]:
# ============================================================
# ZERO-FIELD CHECK (VERY IMPORTANT BASELINE)
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Ensure magnet is OFF initially
magnet.set_current(0.0)
time.sleep(2)

X0 = lockin.data.value['X']
Y0 = lockin.data.value['Y']
R0 = lockin.data.value['R']
B0 = magnet.read_field()

print("\nZERO FIELD BASELINE")
print(f"B = {B0:.3f} mT | X = {X0:.3e} | Y = {Y0:.3e} | R = {R0:.3e}")

b''
b''
b''
b''
b''
b'\x12'
b''
b''
b''
b''
b''
b'\x12'
b''
b''
b''
b''
b'\x12'
Magnet current command sent: 0.000 mA
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b''
b'\x12'

ZERO FIELD BASELINE
B = -4.400 mT | X = 3.992e-08 | Y = -4.402e-08 | R = 6.328e-08


In [12]:
# ==========================================
# PERMANENT CONSTANT CURRENT MODE
# ==========================================

# ❗ SET YOUR DESIRED CURRENT HERE (MAX 1 mA)
I_set = 0.001   # Amperes → 1 mA

# SAFETY CHECK (DO NOT REMOVE)
if abs(I_set) > 0.001:
    raise ValueError("Max allowed current is 1 mA")

# Set constant current
magnet.set_current(I_set, wait_time=2.0)

print("\nELECTROMAGNET STATUS")
print(f"Constant current set to {I_set*1e3:.3f} mA")
print("Current will remain constant until changed or turned OFF")


b''
b''
b''
b''
b'\x12'
b''
b''
b''
b''
b'\x12'
b''
b''
b''
b''
b'\x12'
Magnet current command sent: 1.000 mA

ELECTROMAGNET STATUS
Constant current set to 1.000 mA
Current will remain constant until changed or turned OFF


In [13]:
# MAGNET CURRENT SWEEP (0 → 1 mA)

#currents = np.linspace(0.0, 0.001, 11)   # 0 to 1 mA
#settle_time = 2.0                       # seconds

#results = []

#print("\nStarting magnet current sweep...\n")

#for I in currents:
 #   magnet.set_current(I, wait_time=settle_time)

  #  X = lockin.data.value['X']
  #  Y = lockin.data.value['Y']
  #  R = lockin.data.value['R']
 #   B = magnet.read_field()

  #  results.append({
 #       "current_A": I,
   #     "current_mA": I * 1e3,
  #      "field_mT": B,
  #      "X": X,
   #     "Y": Y,
   #     "R": R,
   #     "timestamp": datetime.now().isoformat()
   # })

  #  print(f"I = {I*1e3:.3f} mA | B = {B:.2f} mT | R = {R:.3e}")

In [14]:
# MAGNET CURRENT SWEEP (0 → 1 mA)

#currents = np.linspace(0.0, 0.001, 11)   # 0 to 1 mA
#settle_time = 2.0                       # seconds

#results = []

#print("\nStarting magnet current sweep...\n")

#for I in currents:
 #   magnet.set_current(I, wait_time=settle_time)

 #   X = lockin.data.value['X']
 #   Y = lockin.data.value['Y']
 #   R = lockin.data.value['R']
  #  B = magnet.read_field()

  #  results.append({
   #     "current_A": I,
   #     "current_mA": I * 1e3,
   #     "field_mT": B,
   #     "X": X,
   #     "Y": Y,
   #     "R": R,
   #     "timestamp": datetime.now().isoformat()
  #  })

   # print(f"I = {I*1e3:.3f} mA | B = {B:.2f} mT | R = {R:.3e}")
    